Importing all libraries

In [1]:
# Import all libraries
import warnings
warnings.filterwarnings("ignore")
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai import ChatMistralAI
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

True

In [3]:
#now loading our chat mistral model and embending model
llm=ChatMistralAI(model="mistral-small-latest",
    temperature=0.7)  # 0.7 for balanced answer 

embeddings=MistralAIEmbeddings(
    model="mistral-embed")

making loader and spliter for pdf and url and you tube url and in return chunk making function

In [4]:
def load_and_split(source_type: str, source: str) -> list:
    """
    Load documents from any source and split into chunks.
    
    Args:
        source_type: "pdf", "url", "youtube", "text"
        source: file path, URL, or plain text
    
    Returns:
        list of document chunks
    """
    
    text_splitter = RecursiveCharacterTextSplitter(#chat spliter use for making split latter by latter and line by line and paragraph by paragraph
        chunk_size=1000,
        chunk_overlap=200
    )
    
    if source_type == "pdf":  #if file is pdf
        loader = PyPDFLoader(source)
        documents = loader.load()
        
    elif source_type == "url": #if its is url
        loader = WebBaseLoader(source)
        documents = loader.load()
        
    elif source_type == "youtube": #if its is url link
        loader = YoutubeLoader.from_youtube_url(source)
        documents = loader.load()
        
    elif source_type == "text": #if its texts or  paragraph
        
        documents = [Document(page_content=source)]
        
    else:
        raise ValueError(f"Invalid source_type: {source_type}. Use 'pdf', 'url', 'youtube' or 'text'")
    
    chunks = text_splitter.split_documents(documents)

    return chunks

now if cloude in pincone server index is not created it creates

In [5]:
# Call the function to load PDF and create chunks
chunks = load_and_split("pdf", "Deeplearning.pdf")
print(f"Total chunks: {len(chunks)}")

Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 104 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignoring wrong pointing object 221 0 (offset 0)
Ignoring wrong pointing object 257 0 (offset 0)
Ignoring wrong pointing object 269 0 (offset 0)
Ignoring wrong pointing object 305 0 (offset 0)
Ignoring wrong pointing object 322 0 (offset 0)
Ignoring wrong pointing object 334 0 (offset 0)
Ignoring wrong pointing object 346 0 (offset 0)
Ignoring wrong pointing object 788 0 (offset 0)
Ignoring wrong pointing object 920 0 (offset 0)
Ignoring wrong pointing object 949 0 (offset 0)
Ignoring wrong pointing object 969 0 (offset 0)
Ignoring wrong pointing object 993 0 (offset 0)
Ignoring wrong pointing object 1029 0 (offset 0)
Ignoring wrong pointing object 1037 0 (offset 0)
Ignoring wrong pointing object 1039 0 (offset 0)
Ignoring wrong pointing object 1041 0 

Total chunks: 24


In [6]:
#for making the index in pinecone server if not exist
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "rag-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Index created!")
else:
    print("Index already exists!")  

Index already exists!


now chunks are converting intto vectors and storing vector into pincone cloude server

In [7]:
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)
print ("chunks are stored in pincone server")

chunks are stored in pincone server


now we make retrivel so they fatch query from vector data base 

In [8]:
# Semantic retriever (Pinecone)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready!")

Retriever ready!


In [9]:
prompt = ChatPromptTemplate.from_template("""
you are Documind :- You are rag chatbot- a helpful AI assistant.
Answer the question based on the context provided.
If answer is not in context, say "I don't have enough information about this in the document."

Context: {context}

Chat History: {chat_history}

Question: {question}

Answer:
""")
print(" Prompt ready!")

 Prompt ready!


making LCEL pipeline structure

In [10]:
def format_docs(docs: list) -> str:
    """Joins all chunks into one string"""
    return "\n\n".join(doc.page_content for doc in docs)

def build_chain(retriever):
    """Builds LCEL pipeline"""
    
    lcel_chain = (
        {
            "context": lambda x: format_docs(retriever.invoke(x["question"])),
            "chat_history": lambda x: x["chat_history"],
            "question": lambda x: x["question"]
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    
    return lcel_chain

chain = build_chain(retriever)
print(" Chain ready!")

 Chain ready!


for creating chat history and hadling message human msg and ai,system,tool msges

In [11]:
chat_history = []

def chat(question: str) -> str:
    """Chat with the document"""
    
    response = chain.invoke({
        "question": question,
        "chat_history": history_str(chat_history)  # ← use new function
    })
    
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    return response

print(" Chat function ready!")

 Chat function ready!


now we test our chat bot

In [12]:
#give score logic if anyhing go wrong use 3/5 retrival answer
# Grade chunks function
def grade_chunks(question: str, chunks: list) -> tuple:
    """
    Grade retrieved chunks out of 5.0
    Returns score and label
    """
    
    chunks_text = format_docs(chunks)
    
    grade_prompt = f"""
    Question: {question}
    
    Retrieved context:
    {chunks_text}
    
    Rate how relevant this context is for answering the question.
    Reply with ONLY a single decimal number between 1.0 and 5.0.
    Use 0.5 steps only: 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0
    
    Decimal number only:
    """
    
    result = llm.invoke(grade_prompt)
    score_text = result.content.strip()
    
    try:
        score = float(score_text[:3])
        score = max(1.0, min(5.0, score))
    except:
        score = 3.0
    
    return score

print(" Grade chunks ready!")

 Grade chunks ready!


In [13]:
#adding label into answer
def get_label(score : float) -> str:
    """
    Convert score to human readable label
    """
    if score > 4.5:
        return "  Excellent Answer "
    elif score >= 4.0:
        return " Good Answer"
    elif score >= 3.0:
        return " Average Answer— retrying..."
    elif score >= 2.0:
        return " Poor Answer— retrying..."
    else:
        return " Very Poor Answer— retrying..."

In [14]:
def history_str(chat_history: list) -> str:
    """Convert chat history list to plain string"""
    history = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history += f"Human: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history += f"AI: {msg.content}\n"
    return history

print("History helper ready!")

History helper ready!


In [15]:
#now we make self healing fuction
def self_healing_chat(question: str) -> tuple:
    """Self-healing chat with chunk grading"""
    
    current_question = question
    
    for attempt in range(1, 4):
        chunks = retriever.invoke(current_question)
        score = grade_chunks(current_question, chunks)
        label = get_label(score)
        
        print(f"Attempt {attempt} → {score}/5.0 → {label}")
        
        if score >= 3.0:
            break
        
        current_question = question + " explain in detail with examples"
    
    response = chain.invoke({
        "question": question,
        "chat_history": history_str(chat_history)
    })
    
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    return response, score, label, attempt

print(" Self-healing chat ready!")
    

 Self-healing chat ready!


In [19]:
response, score, label, attempts = self_healing_chat("What is perceptron?")

print(f"Score: {score}/5.0")
print(f" Label: {label}")
print(f" Attempts: {attempts}")
print(f" Answer:\n{response}")


Attempt 1 → 2.0/5.0 →  Poor Answer— retrying...
Attempt 2 → 2.0/5.0 →  Poor Answer— retrying...
Attempt 3 → 3.5/5.0 →  Average Answer— retrying...
Score: 3.5/5.0
 Label:  Average Answer— retrying...
 Attempts: 3
 Answer:
**Answer:** A perceptron is the simplest type of artificial neuron, introduced by Frank Rosenblatt in 1958. It serves as the foundation for neural networks and deep learning by mimicking how a biological neuron works—taking inputs, processing them, and producing an output.
